<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 03. Regularización: Ridge (L2) vs Lasso (L1)
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/07%20-%20Regression/Para%20Dummies/03_Regresion_Polinomial_y_Regularizacion_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del cuaderno 03 de Regresión. Si el cuaderno principal usó palabras como *PolynomialFeatures*, *Ridge*, *Lasso* o *ElasticNet* y sentiste que iba muy rápido, aquí vamos más despacio.

Al terminar podrás explicar, con tus propias palabras:
1. Cómo una regresión "lineal" puede dibujar curvas usando `PolynomialFeatures`.
2. Por qué las curvas muy flexibles tienden a sobreajustarse.
3. Qué hacen Ridge y Lasso para "poner freno" a un modelo demasiado flexible, y en qué se diferencian.
4. Qué es el hiperparámetro `alpha` y por qué usamos validación cruzada para elegirlo.

---
## 1. De rectas a curvas: `PolynomialFeatures` 🎢

Una regresión lineal común solo puede dibujar líneas rectas (o planos, con varias variables). Pero muchos fenómenos del mundo real no son rectos — por ejemplo, la relación entre la velocidad de un carro y su consumo de gasolina, o entre la dosis de un medicamento y su efecto.

El truco de `PolynomialFeatures` es simple pero poderoso: en vez de darle al modelo solo la variable `x`, también le damos `x²`, `x³`, etc., como si fueran variables nuevas. El modelo sigue siendo "lineal" en el sentido matemático (sigue sumando pesos por variable), pero como ahora tiene curvas disponibles como ingredientes, ¡puede dibujar líneas curvas!

Es como darle a un artista, además de una regla recta, plantillas con curvas de distintas formas: sigue "trazando líneas", pero ahora con más formas disponibles.

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

x = np.array([[1], [2], [3], [4]])

poly = PolynomialFeatures(degree=3, include_bias=False)
x_poly = poly.fit_transform(x)

print("Valores originales de x:")
print(x.flatten())
print("\nNuevas columnas generadas (x, x², x³):")
print(x_poly)
print("\nNombres de las nuevas características:", poly.get_feature_names_out(['x']))

### 🤔 ¿Qué acaba de pasar?

- `PolynomialFeatures(degree=3)` tomó nuestra única columna `x` y generó tres columnas: `x`, `x²` y `x³`.
- Ahora, cuando le pasemos esta matriz ampliada a un `LinearRegression`, el modelo podrá combinar estas tres columnas con sus propios pesos, lo que en la práctica le permite dibujar curvas (parábolas, curvas en forma de S, etc.) en vez de solo líneas rectas.
- Cuantas más potencias agregues (grado más alto), más "flexible" — y más propensa a sobreajustarse — se vuelve la curva, tal como vimos con el estudiante que memoriza en el cuaderno anterior.

---
## 2. El problema de una curva demasiado flexible 🐍

Si le das a un modelo demasiadas potencias (`x`, `x²`, ..., `x¹⁰`), la curva se vuelve tan flexible que puede literalmente **retorcerse para pasar exactamente por cada punto de entrenamiento** — incluyendo el ruido aleatorio de esos datos. El resultado se ve perfecto en el conjunto de entrenamiento, pero es un desastre prediciendo datos nuevos: es la misma idea de sobreajuste que vimos antes, ahora causada por demasiada curvatura en vez de por demasiadas variables.

La solución no es prohibir las curvas — a veces sí las necesitamos — sino ponerle un **freno** a qué tan grandes pueden llegar a ser los pesos (`coef_`) de esas potencias altas. A eso se le llama **Regularización**.

---
## 3. Regularización: poniendo frenos al sobreajuste 🛑

La regularización agrega una "multa" a la fórmula que el modelo intenta minimizar: mientras más grandes sean los pesos (`coef_`), mayor la multa. Esto empuja al modelo a preferir pesos pequeños, a menos que un peso grande realmente valga la pena porque mejora mucho la predicción.

Existen dos "sabores" principales de esa multa:

- **Ridge ($L_2$):** encoge todos los pesos, haciéndolos más pequeños, pero **casi nunca exactamente cero**. Es como pedirle a todos los ingredientes de la receta que usen porciones más modestas, sin eliminar a ninguno del todo.
- **Lasso ($L_1$):** encoge los pesos y, además, **puede llevar a los menos útiles exactamente a cero** — literalmente los elimina de la fórmula. Es como si el chef decidiera sacar por completo los ingredientes que no aportan sabor.

Por eso decimos que Lasso hace "selección automática de variables": simplifica el modelo por sí solo.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

np.random.seed(42)
X = np.sort(np.random.uniform(-3, 3, 40)).reshape(-1, 1)
y = np.sin(X).flatten() + np.random.normal(0, 0.2, 40)

# Comparamos un polinomio de grado 10 sin regularizar vs Ridge vs Lasso
pipe_lin = make_pipeline(PolynomialFeatures(10), StandardScaler(), LinearRegression()).fit(X, y)
pipe_ridge = make_pipeline(PolynomialFeatures(10), StandardScaler(), Ridge(alpha=1.0)).fit(X, y)
pipe_lasso = make_pipeline(PolynomialFeatures(10), StandardScaler(), Lasso(alpha=0.05)).fit(X, y)

coefs_lin = pipe_lin.named_steps['linearregression'].coef_
coefs_ridge = pipe_ridge.named_steps['ridge'].coef_
coefs_lasso = pipe_lasso.named_steps['lasso'].coef_

print(f"Peso máximo -> Sin regularizar: {np.abs(coefs_lin).max():.2f} | Ridge: {np.abs(coefs_ridge).max():.2f} | Lasso: {np.abs(coefs_lasso).max():.2f}")
print(f"Cantidad de pesos EXACTAMENTE en cero con Lasso: {np.sum(coefs_lasso == 0)} de {len(coefs_lasso)}")

### 🤔 ¿Qué acaba de pasar?

- Generamos datos en forma de onda (`sin(x)`) con ruido, y ajustamos un polinomio de grado 10 — una curva muy flexible, propensa a sobreajustarse.
- El modelo **sin regularizar** puede terminar con pesos enormes en las potencias altas, tratando de pasar por cada punto de ruido.
- **Ridge** debería mostrar pesos máximos mucho más pequeños que el modelo sin regularizar — los "frenó" a todos, pero ninguno llega exactamente a cero.
- **Lasso** debería tener varios pesos en **exactamente cero** — literalmente eliminó esas potencias de la fórmula, quedándose solo con las que aportan de verdad.
- Nota que usamos `StandardScaler()` **antes** de Ridge y Lasso: esto es obligatorio, porque estas técnicas son sensibles a la escala de las variables (una variable en millones "pesa" distinto a una en decimales si no se estandarizan primero).

---
## 4. ¿Cómo elijo el freno correcto? El hiperparámetro `alpha` 🎚️

Tanto Ridge como Lasso tienen un control llamado `alpha` que decide **qué tan fuerte es el freno**:

- `alpha` muy pequeño (cercano a 0): casi no hay freno — el modelo se comporta casi como una regresión sin regularizar (riesgo de sobreajuste).
- `alpha` muy grande: freno excesivo — el modelo se vuelve demasiado simple y pierde capacidad de aprender el patrón real (riesgo de subajuste).

¿Cómo encontramos el `alpha` justo? En vez de adivinar, scikit-learn nos da `RidgeCV` y `LassoCV`: prueban automáticamente muchos valores de `alpha` usando validación cruzada (ya verás qué es esto en el próximo cuaderno) y se quedan con el que mejor generaliza.

In [ ]:
from sklearn.linear_model import RidgeCV

alphas_candidatos = np.logspace(-3, 2, 50)

pipe_ridge_cv = make_pipeline(PolynomialFeatures(10), StandardScaler(), RidgeCV(alphas=alphas_candidatos, cv=5))
pipe_ridge_cv.fit(X, y)

mejor_alpha = pipe_ridge_cv.named_steps['ridgecv'].alpha_
print(f"De los {len(alphas_candidatos)} valores probados, el mejor alpha encontrado fue: {mejor_alpha:.5f}")

### 🤔 ¿Qué acaba de pasar?

- `np.logspace(-3, 2, 50)` genera 50 valores de `alpha` distribuidos entre 0.001 y 100 — un rango amplio de posibles "fuerzas de freno".
- `RidgeCV` prueba cada uno de esos 50 valores usando validación cruzada de 5 particiones (`cv=5`) y se queda automáticamente con el que produce mejores predicciones en datos que el modelo no vio durante ese ajuste interno.
- El resultado (`alpha_`) es el freno óptimo encontrado sin que tuviéramos que probarlos uno por uno a mano — esa automatización es una de las razones por las que `RidgeCV` y `LassoCV` son tan usados en la práctica.

---
## 5. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| `PolynomialFeatures` | Convierte una variable `x` en `x, x², x³...` para que un modelo "lineal" pueda dibujar curvas. |
| Regularización | Le agrega una "multa" a los pesos grandes del modelo para frenar el sobreajuste. |
| Ridge (L2) | Encoge todos los pesos, pero ninguno llega exactamente a cero. |
| Lasso (L1) | Encoge los pesos y puede llevar a cero exacto a los menos útiles (selección automática). |
| `alpha` | Qué tan fuerte es el freno: muy bajo = poco freno (sobreajuste), muy alto = demasiado freno (subajuste). |
| `RidgeCV` / `LassoCV` | Prueban muchos valores de `alpha` automáticamente y eligen el mejor. |

➡️ **Siguiente paso:** en el cuaderno [04 - Selección de Modelos, Validación Cruzada y k-NN (Para Dummies)](04_Seleccion_Modelos_Validacion_Cruzada_y_KNN_Dummies.ipynb) descubrirás exactamente cómo funciona esa "validación cruzada" que `RidgeCV` y `LassoCV` usan por debajo, y conocerás un modelo totalmente distinto: k-NN.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
